In [0]:
from pyspark.sql import functions as F

generation_id = "a5f981814ebf7080"
mapping_version = "v1"

In [0]:
bronze_tc = (
    spark.table("ontologysim.bronze.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

silver_tc = (
    spark.table("ontologysim.silver.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

canonical_tc = (
    spark.table("ontologysim.canonical.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

relationships = (
    spark.table("ontologysim.canonical.relationships")
    .filter(F.col("_generation_id") == generation_id)
)

In [0]:
def relevant_columns(df):
    keywords = (
        "component",
        "target",
        "sut",
        "swc",
    )

    return [
        col
        for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in keywords
        )
    ]


print("BRONZE:")
print(relevant_columns(bronze_tc))

print("\nSILVER:")
print(relevant_columns(silver_tc))

print("\nCANONICAL:")
print(relevant_columns(canonical_tc))

BRONZE:
['component_refs', 'target_ecu']

SILVER:
['component_refs', 'target_ecu']

CANONICAL:
['target_ecu', 'target_component']


In [0]:
def profile_columns(df, layer):
    rows = []

    for column in relevant_columns(df):
        non_empty = (
            df
            .filter(
                F.col(column).isNotNull()
                & (
                    F.trim(
                        F.col(column).cast("string")
                    ) != ""
                )
            )
            .count()
        )

        distinct_values = (
            df
            .filter(
                F.col(column).isNotNull()
                & (
                    F.trim(
                        F.col(column).cast("string")
                    ) != ""
                )
            )
            .select(column)
            .distinct()
            .count()
        )

        samples = [
            str(row[0])
            for row in (
                df
                .filter(F.col(column).isNotNull())
                .select(column)
                .distinct()
                .limit(8)
                .collect()
            )
        ]

        rows.append(
            (
                layer,
                column,
                non_empty,
                distinct_values,
                ", ".join(samples),
            )
        )

    return rows


profile_rows = (
    profile_columns(bronze_tc, "BRONZE")
    + profile_columns(silver_tc, "SILVER")
    + profile_columns(canonical_tc, "CANONICAL")
)

profile_df = spark.createDataFrame(
    profile_rows,
    [
        "layer",
        "column",
        "non_empty_rows",
        "distinct_values",
        "samples",
    ],
)

display(profile_df)

layer,column,non_empty_rows,distinct_values,samples
BRONZE,component_refs,1005,99,"AuxiliaryComponent0026, AuxiliaryComponent0008, AuxiliaryComponent0040, AuxiliaryComponent0073, AuxiliaryComponent0055, AuxiliaryComponent0019, AuxiliaryComponent0072, AuxiliaryComponent0082"
BRONZE,target_ecu,1005,3,"HCP5, HCP3, ConMod"
SILVER,component_refs,1005,99,"AuxiliaryComponent0026, AuxiliaryComponent0008, AuxiliaryComponent0040, AuxiliaryComponent0073, AuxiliaryComponent0055, AuxiliaryComponent0019, AuxiliaryComponent0072, AuxiliaryComponent0082"
SILVER,target_ecu,1005,3,"HCP5, HCP3, CONMOD"
CANONICAL,target_ecu,1000,3,"HCP3, HCP5, CONMOD"
CANONICAL,target_component,0,0,


In [0]:
test_case_targets = (
    relationships
    .filter(
        (F.col("source_entity_type") == "TEST_CASE")
        & (
            F.col("relationship_type")
            == "TARGETS"
        )
    )
)

display(
    test_case_targets
    .groupBy(
        "target_entity_type",
        "relationship_status",
    )
    .count()
    .orderBy(
        "target_entity_type",
        "relationship_status",
    )
)

target_entity_type,relationship_status,count
ECU,RESOLVED,1000


In [0]:
display(
    canonical_tc
    .select(
        "source_test_case_id",
        "target_ecu",
        "target_component",
    )
    .limit(30)
)

source_test_case_id,target_ecu,target_component
TC-BT-000018,HCP3,null
TC-BT-000026,HCP5,null
TC-BT-000027,CONMOD,null
TC-BT-000029,HCP5,null
TC-BT-000043,HCP3,null
TC-BT-000066,HCP3,null
TC-BT-000068,HCP3,null
TC-BT-000099,HCP3,null
TC-BT-000106,HCP5,null
TC-BT-000134,CONMOD,null


In [0]:
import uuid

def normalize_expr(column):
    return F.lower(
        F.regexp_replace(
            F.trim(column.cast("string")),
            r"[^A-Za-z0-9]+",
            "",
        )
    )


def write_generation(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_generation_id = '{generation_id}'",
        )
        .saveAsTable(table_name)
    )


tc_refs_df = (
    silver_tc
    .groupBy("test_case_id")
    .agg(
        F.array_sort(
            F.collect_set(
                F.when(
                    F.col("component_refs").isNotNull()
                    & (F.trim("component_refs") != ""),
                    F.trim("component_refs"),
                )
            )
        ).alias("_component_refs"),

        F.max("_source_file").alias("_source_file"),
    )
)


conflicting_refs = (
    tc_refs_df
    .filter(F.size("_component_refs") > 1)
    .count()
)

print("TestCases with conflicting component refs:", conflicting_refs)

if conflicting_refs:
    raise ValueError(
        "Duplicate TestCase records disagree on component_refs"
    )


tc_refs_df = (
    tc_refs_df
    .withColumn(
        "source_component_ref",
        F.element_at("_component_refs", 1),
    )
    .drop("_component_refs")
)

print(
    "Unique TestCases with Component refs:",
    tc_refs_df
    .filter(F.col("source_component_ref").isNotNull())
    .count(),
)

TestCases with conflicting component refs: 0
Unique TestCases with Component refs: 1000


In [0]:
components = (
    spark.table("ontologysim.canonical.components")
    .filter(F.col("_generation_id") == generation_id)
)

aliases = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)


id_lookup = (
    components
    .select(
        normalize_expr(
            F.col("source_component_id")
        ).alias("lookup_key"),
        "canonical_id",
        F.lit(1).alias("priority"),
    )
)


name_lookup = (
    components
    .select(
        normalize_expr(
            F.col("component_name")
        ).alias("lookup_key"),
        "canonical_id",
        F.lit(2).alias("priority"),
    )
)


alias_lookup = (
    aliases
    .filter(
        (F.col("entity_type") == "COMPONENT")
        & (F.col("mapping_status") == "MATCHED")
        & F.col("canonical_id").isNotNull()
    )
    .select(
        F.col("normalized_alias").alias(
            "lookup_key"
        ),
        "canonical_id",
        F.lit(3).alias("priority"),
    )
)


component_lookup = (
    id_lookup
    .unionByName(name_lookup)
    .unionByName(alias_lookup)
    .groupBy("lookup_key")
    .agg(
        F.collect_set("canonical_id").alias("_ids"),
        F.min("priority").alias("_priority"),
    )
    .withColumn(
        "candidate_count",
        F.size("_ids"),
    )
    .withColumn(
        "canonical_component_id",
        F.when(
            F.col("candidate_count") == 1,
            F.element_at("_ids", 1),
        ),
    )
    .withColumn(
        "mapping_method",
        F.when(
            F.col("_priority") == 1,
            F.lit("DIRECT_ID"),
        ).when(
            F.col("_priority") == 2,
            F.lit("NORMALIZED_NAME"),
        ).otherwise(
            F.lit("ALIAS_RULE")
        ),
    )
    .drop("_ids", "_priority")
)

In [0]:
tc_component_resolution = (
    canonical_tc.alias("tc")
    .join(
        tc_refs_df.alias("ref"),
        F.col("tc.source_test_case_id")
        == F.col("ref.test_case_id"),
        "left",
    )
    .withColumn(
        "_lookup_key",
        normalize_expr(
            F.col("ref.source_component_ref")
        ),
    )
    .join(
        component_lookup.alias("lookup"),
        F.col("_lookup_key")
        == F.col("lookup.lookup_key"),
        "left",
    )
    .select(
        F.col("tc.canonical_id").alias(
            "source_canonical_id"
        ),
        F.col("tc.source_test_case_id"),
        F.col("ref.source_component_ref"),
        F.col("ref._source_file"),

        F.col(
            "lookup.canonical_component_id"
        ),

        F.col("lookup.mapping_method"),

        F.when(
            F.col("ref.source_component_ref").isNull(),
            F.lit("UNRESOLVED"),
        ).when(
            F.col("lookup.candidate_count").isNull(),
            F.lit("UNRESOLVED"),
        ).when(
            F.col("lookup.candidate_count") > 1,
            F.lit("AMBIGUOUS"),
        ).otherwise(
            F.lit("MATCHED")
        ).alias("mapping_status"),
    )
)


display(
    tc_component_resolution
    .groupBy(
        "mapping_status",
        "mapping_method",
    )
    .count()
    .orderBy(
        "mapping_status",
        "mapping_method",
    )
)

mapping_status,mapping_method,count
MATCHED,NORMALIZED_NAME,897
UNRESOLVED,null,103


In [0]:
tc_ref_patch = (
    tc_refs_df
    .select(
        F.col("test_case_id").alias(
            "_patch_test_case_id"
        ),
        F.col("source_component_ref").alias(
            "_patch_component"
        ),
    )
)


canonical_tc_fixed = (
    canonical_tc
    .drop("target_component")
    .join(
        tc_ref_patch,
        F.col("source_test_case_id")
        == F.col("_patch_test_case_id"),
        "left",
    )
    .withColumn(
        "target_component",
        F.col("_patch_component"),
    )
    .drop(
        "_patch_test_case_id",
        "_patch_component",
    )
)


write_generation(
    canonical_tc_fixed,
    "ontologysim.canonical.test_cases",
)

print(
    "Canonical TestCases with Component reference:",
    canonical_tc_fixed
    .filter(F.col("target_component").isNotNull())
    .count(),
)

Canonical TestCases with Component reference: 1000


In [0]:
tc_component_aliases = (
    tc_component_resolution
    .select(
        F.col("canonical_component_id").alias(
            "canonical_id"
        ),

        F.lit("COMPONENT").alias(
            "entity_type"
        ),

        F.col("source_component_ref").alias(
            "alias_value"
        ),

        normalize_expr(
            F.col("source_component_ref")
        ).alias("normalized_alias"),

        F.lit("test_management").alias(
            "source_family"
        ),

        F.lit("SOURCE_REFERENCE").alias(
            "alias_type"
        ),

        F.col("mapping_status"),

        F.col("mapping_method"),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.lit(generation_id).alias(
            "_generation_id"
        ),

        "_source_file",
    )
    .dropDuplicates(
        [
            "source_family",
            "normalized_alias",
        ]
    )
)


existing_aliases = (
    aliases
    .filter(
        ~(
            (F.col("entity_type") == "COMPONENT")
            & (
                F.col("source_family")
                == "test_management"
            )
        )
    )
)


updated_aliases = (
    existing_aliases
    .unionByName(tc_component_aliases)
)


write_generation(
    updated_aliases,
    "ontologysim.canonical.aliases",
)

In [0]:
tc_component_relationships = (
    tc_component_resolution
    .filter(
        F.col("source_component_ref").isNotNull()
    )
    .withColumn(
        "relationship_id",
        F.concat(
            F.lit("CAN-REL-"),
            F.upper(
                F.substring(
                    F.sha2(
                        F.concat_ws(
                            "||",
                            F.lit("TEST_CASE"),
                            F.col("source_test_case_id"),
                            F.lit("TARGETS"),
                            F.lit("COMPONENT"),
                            F.col("source_component_ref"),
                            F.lit("test_management"),
                            F.lit(mapping_version),
                        ),
                        256,
                    ),
                    1,
                    16,
                )
            ),
        ),
    )
    .select(
        "relationship_id",

        F.col("source_canonical_id"),

        F.lit("TEST_CASE").alias(
            "source_entity_type"
        ),

        F.lit("TARGETS").alias(
            "relationship_type"
        ),

        F.col("canonical_component_id").alias(
            "target_canonical_id"
        ),

        F.lit("COMPONENT").alias(
            "target_entity_type"
        ),

        F.col("source_component_ref").alias(
            "target_source_reference"
        ),

        F.lit("EXPLICIT").alias(
            "relationship_origin"
        ),

        F.when(
            F.col("mapping_status") == "MATCHED",
            F.lit("RESOLVED"),
        ).otherwise(
            F.col("mapping_status")
        ).alias("relationship_status"),

        "mapping_method",

        F.when(
            F.col("mapping_method") == "DIRECT_ID",
            F.lit(1.0),
        ).when(
            F.col("mapping_method") == "NORMALIZED_NAME",
            F.lit(0.95),
        ).when(
            F.col("mapping_method") == "ALIAS_RULE",
            F.lit(0.99),
        ).alias("mapping_confidence"),

        F.lit("test_management").alias(
            "source_family"
        ),

        F.col("source_test_case_id").alias(
            "source_record_id"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.lit(generation_id).alias(
            "_generation_id"
        ),

        "_source_file",
    )
)


existing_relationships = (
    spark.table(
        "ontologysim.canonical.relationships"
    )
    .filter(F.col("_generation_id") == generation_id)
    .filter(
        ~(
            (F.col("source_entity_type") == "TEST_CASE")
            & (F.col("relationship_type") == "TARGETS")
            & (F.col("target_entity_type") == "COMPONENT")
        )
    )
)


updated_relationships = (
    existing_relationships
    .unionByName(tc_component_relationships)
    .dropDuplicates(["relationship_id"])
)


write_generation(
    updated_relationships,
    "ontologysim.canonical.relationships",
)

In [0]:
resolved_relationships = (
    updated_relationships
    .filter(
        F.col("relationship_status") == "RESOLVED"
    )
)


graph_edges_fixed = (
    resolved_relationships
    .select(
        F.col("relationship_id").alias(
            "edge_id"
        ),

        F.col("source_canonical_id").alias(
            "source_node_id"
        ),

        "relationship_type",

        F.col("target_canonical_id").alias(
            "target_node_id"
        ),

        "relationship_origin",
        "relationship_status",
        "source_family",
        "mapping_version",

        F.to_json(
            F.struct(
                "mapping_method",
                "mapping_confidence",
                "source_record_id",
                "target_source_reference",
            )
        ).alias("properties"),

        "_generation_id",
    )
)


write_generation(
    graph_edges_fixed,
    "ontologysim.graph_ready.edges",
)

In [0]:
canonical_relationships_now = (
    spark.table(
        "ontologysim.canonical.relationships"
    )
    .filter(F.col("_generation_id") == generation_id)
)


tc_component_now = (
    canonical_relationships_now
    .filter(
        (F.col("source_entity_type") == "TEST_CASE")
        & (F.col("relationship_type") == "TARGETS")
        & (F.col("target_entity_type") == "COMPONENT")
    )
)


print("CANONICAL TESTCASE → COMPONENT")

display(
    tc_component_now
    .groupBy("relationship_status")
    .count()
    .orderBy("relationship_status")
)


graph_edges_now = (
    spark.table("ontologysim.graph_ready.edges")
    .filter(F.col("_generation_id") == generation_id)
)


graph_tc_component = (
    graph_edges_now
    .filter(
        (F.col("relationship_type") == "TARGETS")
        & (
            F.col("source_node_id").startswith(
                "CAN-TC-"
            )
        )
        & (
            F.col("target_node_id").startswith(
                "CAN-CMP-"
            )
        )
    )
)


print(
    "Graph TestCase → Component edges:",
    graph_tc_component.count(),
)

print(
    "Total graph edges:",
    graph_edges_now.count(),
)


nodes_now = (
    spark.table("ontologysim.graph_ready.nodes")
    .filter(F.col("_generation_id") == generation_id)
)


missing_targets = (
    graph_tc_component.alias("e")
    .join(
        nodes_now.alias("n"),
        F.col("e.target_node_id")
        == F.col("n.node_id"),
        "left_anti",
    )
    .count()
)


print(
    "Missing Component target nodes:",
    missing_targets,
)

CANONICAL TESTCASE → COMPONENT


relationship_status,count
RESOLVED,1000


Graph TestCase → Component edges: 1000
Total graph edges: 23644
Missing Component target nodes: 0


In [0]:
unresolved_tc_components = (
    tc_component_resolution
    .filter(F.col("mapping_status") == "UNRESOLVED")
    .alias("r")
    .join(
        silver_tc
        .select(
            "test_case_id",
            "target_ecu",
        )
        .dropDuplicates(["test_case_id"])
        .alias("tc"),
        F.col("r.source_test_case_id")
        == F.col("tc.test_case_id"),
        "left",
    )
)

display(
    unresolved_tc_components
    .groupBy(
        "source_component_ref",
        "target_ecu",
    )
    .count()
    .orderBy(
        F.col("count").desc(),
        "source_component_ref",
    )
)

source_component_ref,target_ecu,count
MediaManager,HCP3,17
ProfileManager,HCP3,17
BluetoothManager,HCP3,16
ConnectivityStatus,HCP3,15
BackendService,CONMOD,9
EnergyService,HCP5,9
NetworkDiagnostics,CONMOD,8
VehicleStateService,HCP5,7
WakeCoordinator,HCP5,4
PowerStateManager,HCP5,1


In [0]:
aliases_to_check = [
    "MediaManager",
    "ProfileManager",
    "BluetoothManager",
    "ConnectivityStatus",
    "BackendService",
    "EnergyService",
    "NetworkDiagnostics",
    "VehicleStateService",
    "WakeCoordinator",
    "PowerStateManager",
]

candidate_components = (
    components
    .select(
        "canonical_id",
        "component_name",
        "source_ecu_id",
    )
    .filter(
        F.lower("component_name").rlike(
            "media|profile|bluetooth|connectivity|backend|energy|network|vehicle|wake|power"
        )
    )
    .orderBy(
        "source_ecu_id",
        "component_name",
    )
)

display(candidate_components)

canonical_id,component_name,source_ecu_id
CAN-CMP-A86DEBF6787682C8,BackendCommunicationService,ECU-CONMOD
CAN-CMP-99EB2004479617C7,ConnectivityService,ECU-CONMOD
CAN-CMP-8E95705B4A783727,NetworkDiagnosticsService,ECU-CONMOD
CAN-CMP-DB1BD195E0FECE45,BluetoothConnectionManager,ECU-HCP3
CAN-CMP-B1CF366DEC7F8B20,ConnectivityStatusService,ECU-HCP3
CAN-CMP-9F79C1A5AEC97E29,MediaSessionManager,ECU-HCP3
CAN-CMP-EAD09678DB459E65,UserProfileManager,ECU-HCP3
CAN-CMP-C2F9CF14E6DCD7B1,EnergyManagementService,ECU-HCP5
CAN-CMP-A4C4A48E5AA09FDE,VehiclePowerStateManager,ECU-HCP5
CAN-CMP-1C399A982DF152F5,VehicleStateDistributor,ECU-HCP5


In [0]:
alias_rules = [
    ("MediaManager", "HCP3", "MediaSessionManager"),
    ("ProfileManager", "HCP3", "UserProfileManager"),
    ("BluetoothManager", "HCP3", "BluetoothConnectionManager"),
    ("ConnectivityStatus", "HCP3", "ConnectivityStatusService"),

    ("BackendService", "CONMOD", "BackendCommunicationService"),
    ("NetworkDiagnostics", "CONMOD", "NetworkDiagnosticsService"),

    ("EnergyService", "HCP5", "EnergyManagementService"),
    ("VehicleStateService", "HCP5", "VehicleStateDistributor"),
    ("WakeCoordinator", "HCP5", "WakeSleepCoordinator"),
    ("PowerStateManager", "HCP5", "VehiclePowerStateManager"),
]

alias_rules_df = (
    spark.createDataFrame(
        alias_rules,
        [
            "source_component_ref",
            "target_ecu",
            "canonical_component_name",
        ],
    )
    .withColumn(
        "_alias_key",
        normalize_expr(F.col("source_component_ref")),
    )
    .withColumn(
        "_component_name_key",
        normalize_expr(F.col("canonical_component_name")),
    )
)


component_rule_lookup = (
    components
    .withColumn(
        "_component_name_key",
        normalize_expr(F.col("component_name")),
    )
    .withColumn(
        "_component_ecu",
        F.upper(
            F.regexp_replace(
                F.col("source_ecu_id"),
                r"^ECU-",
                "",
            )
        ),
    )
)


resolved_alias_rules = (
    alias_rules_df.alias("r")
    .join(
        component_rule_lookup.alias("c"),
        (
            F.col("r._component_name_key")
            == F.col("c._component_name_key")
        )
        & (
            F.upper(F.col("r.target_ecu"))
            == F.col("c._component_ecu")
        ),
        "left",
    )
    .select(
        F.col("r.source_component_ref"),
        F.col("r.target_ecu"),
        F.col("r.canonical_component_name"),
        F.col("c.canonical_id").alias(
            "canonical_component_id"
        ),
    )
)

display(resolved_alias_rules)

missing_rules = (
    resolved_alias_rules
    .filter(F.col("canonical_component_id").isNull())
    .count()
)

if missing_rules:
    raise ValueError(
        f"{missing_rules} alias rules could not be resolved"
    )

print("All 10 alias rules validated")

source_component_ref,target_ecu,canonical_component_name,canonical_component_id
MediaManager,HCP3,MediaSessionManager,CAN-CMP-9F79C1A5AEC97E29
ProfileManager,HCP3,UserProfileManager,CAN-CMP-EAD09678DB459E65
BluetoothManager,HCP3,BluetoothConnectionManager,CAN-CMP-DB1BD195E0FECE45
ConnectivityStatus,HCP3,ConnectivityStatusService,CAN-CMP-B1CF366DEC7F8B20
BackendService,CONMOD,BackendCommunicationService,CAN-CMP-A86DEBF6787682C8
NetworkDiagnostics,CONMOD,NetworkDiagnosticsService,CAN-CMP-8E95705B4A783727
EnergyService,HCP5,EnergyManagementService,CAN-CMP-C2F9CF14E6DCD7B1
VehicleStateService,HCP5,VehicleStateDistributor,CAN-CMP-1C399A982DF152F5
WakeCoordinator,HCP5,WakeSleepCoordinator,CAN-CMP-F8504D0ECACF284A
PowerStateManager,HCP5,VehiclePowerStateManager,CAN-CMP-A4C4A48E5AA09FDE


All 10 alias rules validated


In [0]:
tc_ecu_context = (
    silver_tc
    .select(
        F.col("test_case_id").alias(
            "_context_test_case_id"
        ),
        F.upper("target_ecu").alias(
            "_context_ecu"
        ),
    )
    .dropDuplicates(
        ["_context_test_case_id"]
    )
)


rule_lookup = (
    resolved_alias_rules
    .select(
        normalize_expr(
            F.col("source_component_ref")
        ).alias("_rule_alias_key"),

        F.upper("target_ecu").alias(
            "_rule_ecu"
        ),

        F.col("canonical_component_id").alias(
            "_rule_canonical_id"
        ),
    )
)


tc_component_resolution_v11 = (
    tc_component_resolution.alias("r")
    .join(
        tc_ecu_context.alias("ctx"),
        F.col("r.source_test_case_id")
        == F.col("ctx._context_test_case_id"),
        "left",
    )
    .withColumn(
        "_alias_key",
        normalize_expr(
            F.col("r.source_component_ref")
        ),
    )
    .join(
        rule_lookup.alias("rule"),
        (
            F.col("_alias_key")
            == F.col("rule._rule_alias_key")
        )
        & (
            F.col("ctx._context_ecu")
            == F.col("rule._rule_ecu")
        ),
        "left",
    )
    .select(
        F.col("r.source_canonical_id"),
        F.col("r.source_test_case_id"),
        F.col("r.source_component_ref"),
        F.col("r._source_file"),

        F.coalesce(
            F.col("r.canonical_component_id"),
            F.col("rule._rule_canonical_id"),
        ).alias("canonical_component_id"),

        F.when(
            F.col("r.canonical_component_id").isNotNull(),
            F.col("r.mapping_method"),
        ).when(
            F.col("rule._rule_canonical_id").isNotNull(),
            F.lit("ALIAS_RULE"),
        ).alias("mapping_method"),

        F.when(
            F.col("r.canonical_component_id").isNotNull(),
            F.lit("MATCHED"),
        ).when(
            F.col("rule._rule_canonical_id").isNotNull(),
            F.lit("MATCHED"),
        ).otherwise(
            F.col("r.mapping_status"),
        ).alias("mapping_status"),
    )
)


display(
    tc_component_resolution_v11
    .groupBy(
        "mapping_status",
        "mapping_method",
    )
    .count()
    .orderBy(
        "mapping_status",
        "mapping_method",
    )
)

mapping_status,mapping_method,count
MATCHED,ALIAS_RULE,103
MATCHED,NORMALIZED_NAME,897


In [0]:
tc_component_resolution = (
    tc_component_resolution_v11
)

unresolved = (
    tc_component_resolution
    .filter(
        F.col("mapping_status") != "MATCHED"
    )
    .count()
)

print(
    "Unresolved TestCase Component refs:",
    unresolved,
)

if unresolved:
    raise ValueError(
        "TestCase Component references remain unresolved"
    )

print("TestCase Component resolution complete")

Unresolved TestCase Component refs: 0
TestCase Component resolution complete
